# One-Step Decision Router: Read the Model's Answer Before It Speaks

**Route messages with an LLM in one forward pass, then find out whether its confidence deserves your trust.**

This is the companion notebook to the [DiamantAI](https://www.youtube.com/@DiamantAI) video **[The AI That Knows the Answer Before It Speaks [Jev Explained]](https://www.youtube.com/watch?v=Fo2kisJx92Y&list=PLBrpE2PttR2k)**. The film follows one fraud report through a chatbot model and through Jev, then runs the same test on about three thousand real bank messages. This notebook lets you run it on yours.

## Overview

Many agent steps are decisions, not writing: which team gets this ticket, is this comment spam, does this tool call look safe. Most pipelines still ask a chat model to *write* the answer and then parse the text. That is slow, it costs output tokens, and it throws away the one thing you need to act safely: how sure the model was.

Models built only for this kind of decision have started to appear. TypeSafe's **Jev** is one: it returns a choice with a probability and never writes text. Jev is closed, so this notebook does the same move on an open model:

1. **Read the decision at the first step.** Before an LLM writes its first token, it has already scored every token it knows. Read the scores of your options and you have the choice plus a probability, in one forward pass.
2. **Compare it with the chatbot way.** Let the same model think out loud and write its answer, then compare time and accuracy.
3. **Check whether the number is honest (calibration).** Group answers by confidence and count how often each group is right.
4. **Fix it and put a gate on it.** Rescale the confidence on held-out data (temperature scaling), then route automatically only above a threshold and send the rest to a person.

## Detailed Explanation

### Motivation

A routing agent that is wrong 3 times in 10 is fine *if it knows which 3*. Then those cases go to a person and everything else runs untouched. That only works when the confidence number means what it says: when the model says 90%, it should be right about 9 times in 10. A model with that property is **calibrated**. Language models are often far more confident than they are right, so you have to measure it on your own data before trusting it.

### Key Components

- **One-step decision:** one forward pass, then the log-probabilities of the *first token* of each option name, normalised over the options.
- **Written baseline:** the same model with thinking turned on, generating a full answer that we parse.
- **Calibration check:** a reliability table (confidence bucket → accuracy) and Expected Calibration Error (ECE).
- **Temperature scaling:** one number `T` fitted on held-out labelled messages that softens or sharpens the scores.
- **Confidence gate:** the lowest threshold that keeps held-out accuracy above your target. Messages above it route automatically, the rest go to a human.

### Agent Architecture

![One-Step Decision Router](../images/one_step_decision_router.svg)

### Benefits

- **Fast and cheap:** one forward pass and no generated tokens, instead of hundreds.
- **Actionable:** a probability your code can branch on, not a sentence to parse.
- **Honest:** the calibration check tells you, in numbers, when the model's confidence can be used.

### Data

[banking77](https://github.com/PolyAI-LDN/task-specific-datasets) (PolyAI, CC-BY-4.0): 3,080 real customer messages to a bank, each labelled with one of 77 intents. We map the intents to 7 teams, so the task looks like real ticket routing.

## Required Packages

The notebook runs an open model locally with Hugging Face `transformers`. A GPU (Colab T4 is enough) or an Apple-silicon Mac is recommended, and a CPU also works, only slower. No API key is needed.

In [ ]:
!pip install -q torch transformers accelerate pandas matplotlib

## Implementation

### 1. Configuration

`Qwen/Qwen3-1.7B` runs on a laptop or a free Colab GPU. The film used `Qwen/Qwen3-8B`, which is more accurate but needs about 16 GB of memory. `N_MESSAGES` keeps the first run short, and you can raise it to 3080 for the full test set.

In [ ]:
MODEL_ID = "Qwen/Qwen3-1.7B"   # the film used "Qwen/Qwen3-8B"
N_MESSAGES = 400               # up to 3080
N_WRITTEN = 5                  # messages to also run the slow, write-it-out way
SEED = 7

### 2. Load real customer messages

We download the banking77 test split straight from the dataset's GitHub repository.

In [ ]:
import io, random, urllib.request
import pandas as pd

URL = "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/test.csv"
df = pd.read_csv(io.StringIO(urllib.request.urlopen(URL).read().decode("utf-8")))
df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True).head(N_MESSAGES)
print(len(df), "messages")
df.head()

### 3. Map 77 intents to 7 teams

Real routers send a message to a team, not to one of 77 intents. This mapping is ours. Some messages could fairly go to two teams, and that matters later: an honest model should *split* its probability on those.

In [ ]:
TEAMS = ["cards", "payments", "transfers", "topups", "fees", "security", "account"]
MAP = {
 "cards": "card_arrival card_linking card_delivery_estimate card_not_working contactless_not_working getting_virtual_card card_acceptance get_physical_card visa_or_mastercard disposable_card_limits virtual_card_not_working getting_spare_card order_physical_card get_disposable_virtual_card activate_my_card card_about_to_expire apple_pay_or_google_pay card_swallowed supported_cards_and_currencies pin_blocked change_pin",
 "payments": "pending_card_payment declined_card_payment reverted_card_payment? request_refund Refund_not_showing_up transaction_charged_twice pending_cash_withdrawal wrong_amount_of_cash_received atm_support declined_cash_withdrawal",
 "transfers": "cancel_transfer transfer_not_received_by_recipient declined_transfer pending_transfer transfer_timing beneficiary_not_allowed failed_transfer transfer_into_account receiving_money balance_not_updated_after_bank_transfer",
 "topups": "automatic_top_up pending_top_up top_up_limits top_up_reverted topping_up_by_card verify_top_up top_up_by_cash_or_cheque top_up_failed balance_not_updated_after_cheque_or_cash_deposit",
 "fees": "exchange_rate card_payment_wrong_exchange_rate extra_charge_on_statement fiat_currency_support exchange_via_app card_payment_fee_charged wrong_exchange_rate_for_cash_withdrawal exchange_charge cash_withdrawal_charge transfer_fee_charged top_up_by_bank_transfer_charge top_up_by_card_charge",
 "security": "lost_or_stolen_card compromised_card card_payment_not_recognised direct_debit_payment_not_recognised cash_withdrawal_not_recognised lost_or_stolen_phone",
 "account": "age_limit edit_personal_details why_verify_identity unable_to_verify_identity verify_my_identity verify_source_of_funds terminate_account passcode_forgotten country_support",
}
INTENT2TEAM = {intent: team for team, intents in MAP.items() for intent in intents.split()}
df["team"] = df["category"].map(INTENT2TEAM)
assert df["team"].notna().all()
df["team"].value_counts()

### 4. Load the open model

We use the best available device: CUDA, Apple's MPS, or the CPU.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
dtype = torch.float16 if device != "cpu" else torch.float32
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dtype).to(device).eval()
print(MODEL_ID, "on", device, "| vocabulary:", len(tok), "tokens")

### 5. Find the first token of every option

The model scores *tokens*, not words, so we need the token each team name starts with. We include the capitalised and space-prefixed spellings, because the model may start its answer with any of them. Each team must own its own tokens, or two options would share probability.

In [ ]:
FIRST_IDS = {}
for team in TEAMS:
    ids = {tok.encode(form, add_special_tokens=False)[0]
           for form in (team, team.capitalize(), " " + team, " " + team.capitalize())}
    FIRST_IDS[team] = sorted(ids)
owners = [i for ids in FIRST_IDS.values() for i in ids]
assert len(owners) == len(set(owners)), "two teams share a first token; rename an option"
{team: [tok.decode([i]) for i in ids] for team, ids in FIRST_IDS.items()}

### 6. The one-step decision

The prompt asks for the team name only, and thinking is switched off so the very next token is the answer. We run **one forward pass**, take the log-probabilities at the last position, and add up each team's first-token probabilities. We return the team scores (as log-probabilities), the normalised probabilities, and how much of the model's total probability the seven options held.

In [ ]:
import math, time

SYSTEM = ("You route messages from a bank's customers to the team that should handle them. "
          "Teams: " + ", ".join(TEAMS) + ". Answer with the team name only.")

def build_prompt(text, think=False):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": text}]
    return tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=False,
                                   enable_thinking=think)

@torch.no_grad()
def decide(text):
    ids = tok(build_prompt(text), return_tensors="pt").to(device)
    logp = torch.log_softmax(model(**ids).logits[0, -1].float(), dim=-1)
    scores = {t: torch.logsumexp(logp[FIRST_IDS[t]], dim=0).item() for t in TEAMS}
    mass = sum(math.exp(s) for s in scores.values())
    probs = {t: math.exp(s) / mass for t, s in scores.items()}
    return scores, probs, mass

## Usage Example

### Try it on one message

This is the message the film follows. Look at the bars *before* the model has written anything.

In [ ]:
text = "Is there an option to top up a with cheque?"
start = time.perf_counter()
scores, probs, mass = decide(text)
print(f"{(time.perf_counter() - start) * 1000:.0f} ms | the seven options held {mass:.4f} of all next-token probability")
for team, p in sorted(probs.items(), key=lambda kv: -kv[1]):
    print(f"{team:>10} {p:8.4f} {'#' * int(p * 40)}")

### Route a batch

Now the same thing on every message. We keep the scores so we can recalibrate later without running the model again.

In [ ]:
rows = []
decide("warm up")
for text, truth in zip(df["text"], df["team"]):
    start = time.perf_counter()
    scores, probs, mass = decide(text)
    pred = max(probs, key=probs.get)
    rows.append({"text": text, "truth": truth, "pred": pred, "conf": probs[pred],
                 "ms": (time.perf_counter() - start) * 1000, "mass": mass,
                 **{f"s_{t}": scores[t] for t in TEAMS}})
res = pd.DataFrame(rows)
print(f"accuracy {(res.pred == res.truth).mean():.1%} | median {res.ms.median():.0f} ms per message")
res[["text", "truth", "pred", "conf"]].head(10)

## Comparison

### The chatbot way: let it write, then parse

The same model, with thinking on, writes out its reasoning and then its answer. We time it and check whether the written answer matches the one-step pick.

In [ ]:
import re

@torch.no_grad()
def write_answer(text, max_new_tokens=1024):
    ids = tok(build_prompt(text, think=True), return_tensors="pt").to(device)
    start = time.perf_counter()
    out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False)
    seconds = time.perf_counter() - start
    new = out[0, ids["input_ids"].shape[1]:]
    reply = tok.decode(new, skip_special_tokens=True)
    final = re.sub(r"(?s)<think>.*?</think>", "", reply).lower()
    said = next((t for t in TEAMS if t in final.replace("-", "").replace(" ", "")), None)
    return said, seconds, len(new), reply

written = []
for _, r in res.head(N_WRITTEN).iterrows():
    said, seconds, n_tokens, reply = write_answer(r.text)
    written.append({"text": r.text, "truth": r.truth, "one_step": r.pred, "written": said,
                    "seconds": seconds, "tokens": n_tokens})
    print(f"{seconds:5.1f}s {n_tokens:4d} tokens | one step: {r.pred:>9} | written: {str(said):>9} | truth: {r.truth}")
w = pd.DataFrame(written)
print(f"\none step: {res.ms.head(N_WRITTEN).median() / 1000:.2f}s median | written: {w.seconds.median():.1f}s median, "
      f"{w.tokens.median():.0f} tokens | agreement {(w.one_step == w.written).mean():.0%}")

### Is its confidence honest? The calibration check

Put the answers in groups by how sure the model was, and count how often each group was right. For a calibrated model, the two columns match.

In [ ]:
import numpy as np

BINS = [0.0, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99, 1.0001]

def reliability(conf, correct):
    conf, correct = np.asarray(conf), np.asarray(correct, dtype=float)
    table, ece = [], 0.0
    for lo, hi in zip(BINS[:-1], BINS[1:]):
        m = (conf >= lo) & (conf < hi)
        if m.any():
            table.append({"confidence": f"{lo:.2f}-{min(hi, 1):.2f}", "n": int(m.sum()),
                          "mean confidence": conf[m].mean(), "accuracy": correct[m].mean()})
            ece += m.sum() * abs(conf[m].mean() - correct[m].mean())
    return pd.DataFrame(table), ece / len(conf)

table, ece = reliability(res.conf, res.pred == res.truth)
print(f"Expected Calibration Error: {ece:.3f}  (0 is perfectly honest)")
table.round(3)

### Look at the most confident mistakes

These are the dangerous ones. Software acts on a tall bar.

In [ ]:
wrong = res[res.pred != res.truth].sort_values("conf", ascending=False)
wrong[["text", "truth", "pred", "conf"]].head(10)

### Make the number honest: temperature scaling

We split the labelled messages in half. On the first half we find the one temperature `T` that makes the probabilities most honest (lowest negative log-likelihood). Then we measure the second half, which the fit never saw. `T > 1` means the model was overconfident.

In [ ]:
S = res[[f"s_{t}" for t in TEAMS]].to_numpy()
y = np.array([TEAMS.index(t) for t in res.truth])
half = len(res) // 2

def softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def nll(T, s, labels):
    p = softmax(s / T)
    return -np.log(p[np.arange(len(labels)), labels] + 1e-12).mean()

temps = np.exp(np.linspace(np.log(0.5), np.log(50), 400))
T = temps[np.argmin([nll(t, S[:half], y[:half]) for t in temps])]

p_test = softmax(S[half:] / T)
conf_after = p_test.max(axis=1)
pred_after = p_test.argmax(axis=1)
correct_test = pred_after == y[half:]
_, ece_before = reliability(res.conf[half:], (res.pred == res.truth)[half:])
table_after, ece_after = reliability(conf_after, correct_test)
print(f"T = {T:.2f} | ECE on unseen half: before {ece_before:.3f} -> after {ece_after:.3f}")
table_after.round(3)

### Plot it: the reliability diagram

A calibrated model sits on the diagonal. Points below it are overconfident.

In [ ]:
import matplotlib.pyplot as plt

_, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], "--", color="grey", label="honest")
before, _ = reliability(res.conf[half:], (res.pred == res.truth)[half:])
ax.plot(before["mean confidence"], before["accuracy"], "o-", label="raw model")
ax.plot(table_after["mean confidence"], table_after["accuracy"], "s-", label=f"after T = {T:.1f}")
ax.set_xlabel("how sure the model said it was")
ax.set_ylabel("how often it was right")
ax.set_xlim(0, 1.02); ax.set_ylim(0, 1.02); ax.legend()
plt.show()

### The confidence gate

Pick the lowest threshold whose routed messages reach the target accuracy on the calibration half. The target is a business choice: how often can an automatic route be wrong before a person should have looked? Then check it on the unseen half: how many messages route automatically, and how often those are right.

In [ ]:
TARGET = 0.70   # a business choice; the small default model tops out near here, with Qwen3-8B try 0.90

p_cal = softmax(S[:half] / T)
conf_cal, correct_cal = p_cal.max(axis=1), p_cal.argmax(axis=1) == y[:half]

# the trade-off on the calibration half: a higher gate routes fewer messages, more of them right
trade = []
for th in np.round(np.arange(0.30, 0.96, 0.05), 2):
    m = conf_cal >= th
    if m.sum() >= 10:
        trade.append({"gate": th, "routed automatically": m.mean(), "accuracy of routed": correct_cal[m].mean()})
trade = pd.DataFrame(trade)
print(trade.round(3).to_string(index=False))

ok = trade[trade["accuracy of routed"] >= TARGET]
if ok.empty:
    print(f"\nNo gate reaches {TARGET:.0%} on this model: send everything to a person, or use a bigger model.")
else:
    gate = ok["gate"].min()
    m = conf_after >= gate
    print(f"\ngate {gate:.2f} | unseen half: {m.mean():.0%} routed automatically at {correct_test[m].mean():.1%} accuracy, "
          f"{(~m).mean():.0%} sent to a person (every message routed: {correct_test.mean():.1%})")

## Additional Considerations

- **Your mapping is part of the result.** Messages that fit two teams cap the accuracy. A well-calibrated model shows this as a *split* probability, which is exactly the signal the gate uses.
- **Option names matter.** Give each option a distinct first token (the assertion in step 5 checks this). For long option lists, use letters or short codes in the prompt.
- **Temperature scaling fixes the scale, not the ranking.** It makes the confidence honest but does not change which team is on top. For better picks use a bigger model, a clearer prompt, or a few labelled examples in the prompt.
- **Re-check after every change.** A new model, prompt or kind of traffic changes the calibration. Keep a few hundred labelled messages and re-run this check.
- **Hosted APIs:** any API that returns token log-probabilities (for example `logprobs` with `top_logprobs`) supports the same one-step read. Purpose-built decision models such as TypeSafe's Jev return a choice and a probability directly and say they are trained for calibrated confidence. Either way, the test in this notebook is how you verify that claim on your own data.

## References

- banking77: Casanueva et al., *Efficient Intent Detection with Dual Sentence Encoders* (2020), [dataset](https://github.com/PolyAI-LDN/task-specific-datasets), CC-BY-4.0.
- Guo et al., *On Calibration of Modern Neural Networks* (ICML 2017): temperature scaling and ECE.
- Kahneman, *Thinking, Fast and Slow* (2011): the System 1 / System 2 idea behind the name "System One model".
- TypeSafe AI, [Introducing System One Models & Jev](https://typesafe.ai/blog/introducing-system-one-models-and-jev).
- Qwen team, [Qwen3](https://huggingface.co/Qwen/Qwen3-1.7B) open-weight models.